# Process Expression Notebook

imports + paths + file check

In [ ]:
from pathlib import Path
import os

# set project root = parent of /notebooks
PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("CWD set to:", Path.cwd())

WindowsPath('c:/Users/josha/Codeacademy Projects/Data Scientist Portfolio Project/notebooks')

In [1]:
from pathlib import Path
import re
import pandas as pd

RAW = Path("data/raw")
OUT = Path("data/processed")
OUT.mkdir(parents=True, exist_ok=True)

expr_path = RAW / "OmicsExpressionProteinCodingGenesTPMLogp1.csv"
expr_path.exists(), expr_path

(False, WindowsPath('data/raw/OmicsExpressionProteinCodingGenesTPMLogp1.csv'))

helper to clean gene column names

In [2]:
def clean_gene_cols(cols):
    """
    Convert: 'TSPAN6 (7105)' -> 'TSPAN6'
    Keep uniqueness by appending '__2', '__3', etc. if needed.
    """
    cleaned = []
    seen = {}
    pat = re.compile(r"^(.*)\s+\(\d+\)$")

    for c in cols:
        m = pat.match(c)
        base = (m.group(1) if m else c).strip()
        seen[base] = seen.get(base, 0) + 1
        cleaned.append(base if seen[base] == 1 else f"{base}__{seen[base]}")
    return cleaned

read + inspect

In [3]:
df = pd.read_csv(expr_path)
df.shape, df.columns[:5]

FileNotFoundError: [Errno 2] No such file or directory: 'data\\raw\\OmicsExpressionProteinCodingGenesTPMLogp1.csv'

standardize ID + clean gene cols

In [ ]:
id_col = df.columns[0]  # usually 'Unnamed: 0'
df = df.rename(columns={id_col: "depmap_id"})

gene_cols = [c for c in df.columns if c != "depmap_id"]
cleaned = clean_gene_cols(gene_cols)

feature_map = pd.DataFrame({"original": gene_cols, "feature": cleaned})
feature_map.head()

save processed outputs

In [ ]:
feature_map.to_csv(OUT / "expression_feature_map.csv", index=False)

df.columns = ["depmap_id"] + cleaned
df.to_parquet(OUT / "expression_tpm_logp1.parquet", index=False)

print("Wrote:", OUT / "expression_tpm_logp1.parquet")
print("Wrote:", OUT / "expression_feature_map.csv")
print("Rows:", f"{df.shape[0]:,}", "| Features:", f"{df.shape[1]-1:,}")
df.head()